# Module 10 - 실습 3: AST 기반 코드 압축

## 학습 목표
- Python AST를 사용하여 코드 구조를 파싱할 수 있다
- 주석/docstring을 제거하여 코드를 압축할 수 있다
- 클래스/함수 시그니처만 추출하여 토큰을 절약할 수 있다

## 참조 자료
- 학습 문서: `docs/part4-production/10-resource-optimization.md` (섹션 2.5)

## 개념 요약

**압축 레벨**:
```
Level 0: 전체 코드 (1000 토큰)
Level 1: import 제거 + 주석 제거 (700 토큰)
Level 2: 함수/클래스 시그니처 + docstring만 (300 토큰)
Level 3: 클래스/함수 선언 목록만 (100 토큰)
```

AST(Abstract Syntax Tree)를 사용하면 코드 구조를 분석하고
필요한 부분만 추출할 수 있습니다.

In [ ]:
import sys
sys.path.insert(0, '..')

import ast
from dataclasses import dataclass

print("AST 코드 압축 실습 준비 완료")

## 실습 1: AST 기초 - 코드 파싱

`ast.parse()`로 Python 코드를 파싱하고 구조를 탐색하세요.

In [ ]:
sample_code = '''
class Calculator:
    """계산기 클래스."""
    
    def add(self, a: int, b: int) -> int:
        """두 수를 더합니다."""
        # 덧셈 수행
        result = a + b
        return result
    
    def multiply(self, a: int, b: int) -> int:
        return a * b
'''

# TODO: 여기에 코드를 작성하세요
# 힌트 1: tree = ast.parse(sample_code) 로 파싱
# 힌트 2: ast.walk(tree) 로 모든 노드를 순회
# 힌트 3: isinstance(node, ast.ClassDef) 로 클래스 노드 확인
# 힌트 4: isinstance(node, ast.FunctionDef) 로 함수 노드 확인

# tree = ...
classes = []
functions = []

# TODO: tree를 순회하며 클래스명과 함수명을 수집

print(f"발견된 클래스: {classes}")
print(f"발견된 함수: {functions}")

In [ ]:
# 검증 셀
assert "Calculator" in classes, f"Calculator 클래스를 찾아야 합니다. 발견: {classes}"
assert "add" in functions, f"add 함수를 찾아야 합니다. 발견: {functions}"
assert "multiply" in functions, f"multiply 함수를 찾아야 합니다. 발견: {functions}"
print("✅ 실습 1 완료! AST 파싱이 올바르게 작동합니다.")

## 실습 2: PythonCodeCompressor 클래스 구현

클래스와 함수 시그니처를 추출하는 `PythonCodeCompressor`를 구현하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: extract_signatures()는 ast.walk()로 ClassDef와 FunctionDef를 찾습니다
# 힌트 2: 클래스 선언: f"class {node.name}({bases}):"
# 힌트 3: 함수 선언: f"    def {node.name}({args}){returns}:"
# 힌트 4: ast.get_docstring(node)으로 docstring 추출
# 힌트 5: ast.unparse(arg.annotation)으로 타입 어노테이션 추출

class PythonCodeCompressor:
    """Python 코드의 구조적 압축.
    
    AST를 사용하여 클래스/함수 시그니처만 추출합니다.
    """
    
    def extract_signatures(self, source_code: str) -> str:
        """Python 코드에서 클래스와 함수 시그니처를 추출합니다.
        
        Args:
            source_code: Python 소스 코드 문자열
            
        Returns:
            시그니처만 포함된 요약 문자열
        """
        try:
            tree = ast.parse(source_code)
        except SyntaxError:
            return source_code  # 파싱 실패 시 원본 반환
        
        lines = []
        # TODO: tree를 순회하며 ClassDef와 FunctionDef 처리
        
        return "\n".join(lines)
    
    def _get_args(self, node) -> str:
        """함수 인자를 문자열로 반환합니다."""
        # TODO: node.args.args를 순회하며 "arg: type" 형태로 반환
        args = []
        for arg in node.args.args:
            annotation = ""
            if arg.annotation:
                annotation = f": {ast.unparse(arg.annotation)}"
            args.append(f"{arg.arg}{annotation}")
        return ", ".join(args)
    
    def _get_return_annotation(self, node) -> str:
        """반환 타입 어노테이션을 반환합니다."""
        if node.returns:
            return f" -> {ast.unparse(node.returns)}"
        return ""

In [ ]:
# 테스트
compressor = PythonCodeCompressor()
summary = compressor.extract_signatures(sample_code)
print("추출된 시그니처:")
print(summary)

In [ ]:
# 검증 셀
assert "Calculator" in summary, "클래스명이 포함되어야 합니다"
assert "add" in summary, "add 함수가 포함되어야 합니다"
assert "multiply" in summary, "multiply 함수가 포함되어야 합니다"
assert "result = a + b" not in summary, "함수 본문은 포함되지 않아야 합니다"
assert len(summary) < len(sample_code), "요약이 원본보다 짧아야 합니다"
print(f"원본 크기: {len(sample_code)}자")
print(f"요약 크기: {len(summary)}자")
print(f"압축률: {(1 - len(summary)/len(sample_code))*100:.0f}%")
print("✅ 실습 2 완료! PythonCodeCompressor가 올바르게 작동합니다.")

## 실습 3: 압축 효과 측정

대용량 코드에 압축을 적용하고 토큰 절감 효과를 측정하세요.

In [ ]:
large_code = '''
import logging
import time
from pathlib import Path

logger = logging.getLogger(__name__)


class DataProcessor:
    """데이터를 처리하는 클래스."""

    def __init__(self, config: dict):
        self.config = config
        self.logger = logging.getLogger(__name__)
        self.cache = {}
        self._initialized = False

    def process(self, data: list[dict]) -> list[dict]:
        """데이터를 처리합니다."""
        results = []
        for item in data:
            validated = self._validate(item)
            if validated:
                transformed = self._transform(validated)
                results.append(transformed)
        self.logger.info(f"Processed {len(results)} items")
        return results

    def _validate(self, item: dict) -> dict | None:
        """항목을 검증합니다."""
        if "id" not in item:
            self.logger.warning("Missing id field")
            return None
        if "value" not in item:
            self.logger.warning("Missing value field")
            return None
        return item

    def _transform(self, item: dict) -> dict:
        """항목을 변환합니다."""
        return {
            "id": item["id"],
            "processed_value": item["value"] * 2,
            "timestamp": time.time(),
        }
'''

# TODO: compressor를 사용하여 large_code를 압축하고 토큰 수 비교
# 힌트: tiktoken 없이도 len()으로 간단히 비교 가능

compressor = PythonCodeCompressor()
# summary = compressor.extract_signatures(large_code)
summary = None  # TODO: 실제 압축 수행

print(f"원본 코드 크기: {len(large_code)}자")
print(f"압축 후 크기: {len(summary) if summary else 0}자")

In [ ]:
# 검증 셀
assert summary is not None, "extract_signatures()를 호출하세요"
assert len(summary) < len(large_code) * 0.7, "압축 후 크기가 원본의 70% 미만이어야 합니다"
assert "DataProcessor" in summary, "클래스명이 포함되어야 합니다"
assert "self.cache = {}" not in summary, "구현 코드가 포함되지 않아야 합니다"

compression_ratio = (1 - len(summary)/len(large_code)) * 100
print(f"압축률: {compression_ratio:.0f}%")
print("압축된 코드:")
print(summary)
print("✅ 실습 3 완료! AST 기반 코드 압축이 올바르게 작동합니다.")

## 정리

이번 실습에서 배운 내용:
1. `ast.parse()`로 Python 코드를 AST로 파싱할 수 있다
2. `ast.walk()`로 AST를 순회하며 클래스/함수를 찾을 수 있다
3. 시그니처만 추출하면 코드를 70~80% 압축하여 LLM 토큰을 절약할 수 있다

## 다음 모듈
- **Module 11**: `module_11_quality_assurance/` - 품질 보증